In [ ]:
# Cell 1: Install core ML, Graph, and LLM frameworks
!pip install -q pandas numpy scikit-learn xgboost networkx \
    transformers accelerate datasets requests matplotlib seaborn
print("Colab environment dependencies installed successfully.")

Colab environment dependencies installed successfully.


In [ ]:
# Cell 2: Imports and random state
import os
import re
import json
import warnings
import requests
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from datasets import load_dataset
from transformers import pipeline

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment and seed configured.")

Environment and seed configured.


In [ ]:
# Cell 3: Download and train Credit Card Fraud model
print("--- [Modality 1: Financial Transactions] ---")
url = "https://zenodo.org/records/7395559/files/creditcard.csv"
output_path = "/content/creditcard.csv"

if not os.path.exists(output_path):
    print("Downloading Credit Card Fraud dataset from Zenodo mirror...")
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(output_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Download complete.")

fraud_df = pd.read_csv(output_path)
print(f"Loaded financial records: {fraud_df.shape}")
print(f"Class distribution:\n{fraud_df['Class'].value_counts()}")

# Features & Target
X_fraud = fraud_df.drop(columns=["Class", "Time"])
y_fraud = fraud_df["Class"]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud, y_fraud, test_size=0.20, stratify=y_fraud, random_state=RANDOM_STATE
)

# Train XGBoost on Financial Anomaly Signals
fraud_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=(len(y_train_f) - sum(y_train_f)) / sum(y_train_f),
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
fraud_model.fit(X_train_f, y_train_f)

fraud_prob = fraud_model.predict_proba(X_test_f)[:, 1]
print(f"Financial Model PR-AUC: {average_precision_score(y_test_f, fraud_prob):.4f}")
print(f"Financial Model ROC-AUC: {roc_auc_score(y_test_f, fraud_prob):.4f}")

# Standardize to Common Event Schema
fraud_events = pd.DataFrame({
    "event_id": ["TX_" + str(i) for i in X_test_f.index],
    "event_type": "financial_transaction",
    "entity_id": ["ENTITY_" + str(i % 300) for i in range(len(X_test_f))], # Synthetic entity binding
    "risk_score": fraud_prob,
    "source": "financial_fraud_model"
})
print(f"Normalized Financial Events: {len(fraud_events)}")

--- [Modality 1: Financial Transactions] ---
Download complete.
Loaded financial records: (284807, 31)
Class distribution:
Class
0    284315
1       492
Name: count, dtype: int64
Financial Model PR-AUC: 0.8303
Financial Model ROC-AUC: 0.9707
Normalized Financial Events: 56962


In [ ]:
# Cell 4: Ingest and train Phishing URL classifier via Hugging Face
print("\n--- [Modality 2: Phishing & Malicious URLs] ---")
print("Streaming Phishing URL dataset from Hugging Face...")
url_ds = load_dataset("pirocheto/phishing-url", split="train[:20000]")
url_df = url_ds.to_pandas()

# Identify columns
url_col = "url" if "url" in url_df.columns else url_df.columns[0]
label_col = "status" if "status" in url_df.columns else ("label" if "label" in url_df.columns else url_df.columns[1])

# Convert binary labels (phishing = 1, legitimate = 0)
if url_df[label_col].dtype == object:
    url_df["target"] = url_df[label_col].apply(lambda x: 1 if str(x).lower() in ["phishing", "bad", "malicious", "1"] else 0)
else:
    url_df["target"] = url_df[label_col].astype(int)

X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    url_df[url_col].astype(str), url_df["target"], test_size=0.20, stratify=url_df["target"], random_state=RANDOM_STATE
)

# Character N-gram TF-IDF Vectorizer
url_vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2, max_features=25000)
X_train_u_vec = url_vectorizer.fit_transform(X_train_u)
X_test_u_vec = url_vectorizer.transform(X_test_u)

url_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
url_model.fit(X_train_u_vec, y_train_u)

url_prob = url_model.predict_proba(X_test_u_vec)[:, 1]
print(f"URL Model PR-AUC: {average_precision_score(y_test_u, url_prob):.4f}")

# Standardize to Common Event Schema
url_events = pd.DataFrame({
    "event_id": ["URL_" + str(i) for i in range(len(X_test_u))],
    "event_type": "url_event",
    "entity_id": ["ENTITY_" + str(i % 300) for i in range(len(X_test_u))], # Cross-domain entity binding
    "risk_score": url_prob,
    "source": "phishing_url_model"
})
print(f"Normalized URL Events: {len(url_events)}")


--- [Modality 2: Phishing & Malicious URLs] ---
Streaming Phishing URL dataset from Hugging Face...


README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

data/train.parquet: reconstructing file:   0%|          |  0.00B /  789kB            

data/train.parquet: downloading bytes:           |  0.00B            

data/test.parquet: reconstructing file:   0%|          |  0.00B /  431kB            

data/test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7658 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3772 [00:00<?, ? examples/s]

URL Model PR-AUC: 0.9696
Normalized URL Events: 1532


In [ ]:
# Cell 5: Ingest and train Network Intrusion Model on UNSW-NB15
print("\n--- [Modality 3: Cyber Network Telemetry] ---")
print("Streaming UNSW-NB15 network flow records from Hugging Face...")
net_ds = load_dataset("Mouwiya/UNSW-NB15", split="train[:25000]")
net_df = net_ds.to_pandas()
net_df.columns = [c.strip().lower() for c in net_df.columns]

label_target = next((c for c in ["label", "is_attack", "attack"] if c in net_df.columns), net_df.columns[-1])
drop_cols = [label_target, "attack_cat", "id", "srcip", "dstip", "saddr", "daddr"]
X_net = net_df.drop(columns=[c for c in drop_cols if c in net_df.columns], errors="ignore")
y_net = net_df[label_target].astype(int)

# One-hot encode string features (proto, service, state)
X_net = pd.get_dummies(X_net)
X_net = X_net.replace([np.inf, -np.inf], np.nan).fillna(0)

X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(
    X_net, y_net, test_size=0.20, stratify=y_net, random_state=RANDOM_STATE
)

net_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
net_model.fit(X_train_n, y_train_n)

net_prob = net_model.predict_proba(X_test_n)[:, 1]
print(f"Network Model PR-AUC: {average_precision_score(y_test_n, net_prob):.4f}")

# Standardize to Common Event Schema
net_events = pd.DataFrame({
    "event_id": ["NET_" + str(i) for i in range(len(X_test_n))],
    "event_type": "network_event",
    "entity_id": ["ENTITY_" + str(i % 300) for i in range(len(X_test_n))],
    "risk_score": net_prob,
    "source": "network_behavior_model"
})
print(f"Normalized Network Events: {len(net_events)}")


--- [Modality 3: Cyber Network Telemetry] ---
Streaming UNSW-NB15 network flow records from Hugging Face...


README.md:   0%|          | 0.00/5.33k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  124MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  106MB            

data/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2280090 [00:00<?, ? examples/s]

Network Model PR-AUC: 0.9927
Normalized Network Events: 5000


In [ ]:
# Cell 6: Fuse all standardized event streams
print("\n--- [Multi-Model Fusion Layer] ---")
all_events = pd.concat([fraud_events, url_events, net_events], ignore_index=True)
print(f"Total Unified Multi-Modal Events: {len(all_events)}")
print(all_events["event_type"].value_counts())

# Domain Weighting Configuration
MODEL_WEIGHTS = {
    "network_event": 0.40,
    "financial_transaction": 0.35,
    "url_event": 0.25
}

all_events["weighted_score"] = all_events["risk_score"] * all_events["event_type"].map(MODEL_WEIGHTS)

# Group by Entity to extract cross-domain correlated risk
entity_risk_summary = all_events.groupby("entity_id").agg(
    fused_risk_score=("weighted_score", "sum"),
    max_individual_score=("risk_score", "max"),
    total_events=("event_id", "count"),
    distinct_sources=("source", lambda x: list(set(x))),
    source_count=("source", "nunique")
).reset_index().sort_values(by=["source_count", "fused_risk_score"], ascending=False)

print("\nTop 10 High-Risk Multi-Domain Entities:")
display(entity_risk_summary.head(10))


--- [Multi-Model Fusion Layer] ---
Total Unified Multi-Modal Events: 63494
event_type
financial_transaction    56962
network_event             5000
url_event                 1532
Name: count, dtype: int64

Top 10 High-Risk Multi-Domain Entities:


,entity_id,fused_risk_score,max_individual_score,total_events,distinct_sources,source_count
4,ENTITY_101,3.517646,0.999848,212,"[financial_fraud_model, network_behavior_model...",3
262,ENTITY_65,3.005407,0.992590,212,"[financial_fraud_model, network_behavior_model...",3
29,ENTITY_124,2.937806,0.999518,212,"[financial_fraud_model, network_behavior_model...",3
234,ENTITY_4,2.924416,0.999694,213,"[financial_fraud_model, network_behavior_model...",3
244,ENTITY_49,2.904023,0.998957,212,"[financial_fraud_model, network_behavior_model...",3
27,ENTITY_122,2.893647,0.998762,212,"[financial_fraud_model, network_behavior_model...",3
187,ENTITY_267,2.878476,0.996801,210,"[financial_fraud_model, network_behavior_model...",3
74,ENTITY_165,2.832589,0.999672,212,"[financial_fraud_model, network_behavior_model...",3
212,ENTITY_29,2.761518,0.995695,213,"[financial_fraud_model, network_behavior_model...",3
213,ENTITY_290,2.674778,0.999612,210,"[financial_fraud_model, network_behavior_model...",3


In [ ]:
# Cell 7: Build in-memory multi-modal intelligence graph
print("\n--- [Relational Graph Construction] ---")
G = nx.MultiDiGraph()

# Add Nodes and Edges
for _, row in all_events.iterrows():
    entity = row["entity_id"]
    event = row["event_id"]

    G.add_node(entity, node_type="entity")
    G.add_node(
        event,
        node_type="event",
        event_type=row["event_type"],
        risk_score=float(row["risk_score"]),
        source=row["source"]
    )
    G.add_edge(entity, event, relation="GENERATED_EVENT")

# Connect events that share the same entity identity
entity_event_map = defaultdict(list)
for _, row in all_events.iterrows():
    entity_event_map[row["entity_id"]].append(row["event_id"])

for entity, ev_list in entity_event_map.items():
    if len(ev_list) > 1:
        # Link sequential events within the entity
        for i in range(min(5, len(ev_list) - 1)):
            G.add_edge(ev_list[i], ev_list[i+1], relation="CO_OCCURRING")

print(f"Graph Construction Complete: {G.number_of_nodes()} nodes, {G.number_of_edges()} relationships.")

# Connected Component Analysis
undirected_G = G.to_undirected()
components = sorted(list(nx.connected_components(undirected_G)), key=len, reverse=True)
print(f"Total Structural Clusters: {len(components)}")
print(f"Largest Multi-Modal Cluster Size: {len(components[0])} nodes")


--- [Relational Graph Construction] ---
Graph Construction Complete: 63794 nodes, 64994 relationships.
Total Structural Clusters: 300
Largest Multi-Modal Cluster Size: 214 nodes


In [ ]:
# # Cell 8: Controlled MCP Tool Interface and Local LLM Investigation
# import json
# import torch
# from transformers import pipeline

# print("\n--- [Controlled MCP & LLM Investigation Engine] ---")

# class MockMCPToolServer:
#     """Read-only tool interface restricting LLM access to structured graph queries."""
#     def __init__(self, events_df, graph):
#         self.events_df = events_df
#         self.graph = graph

#     def get_entity_events(self, entity_id: str) -> str:
#         subset = self.events_df[self.events_df["entity_id"] == entity_id]
#         if subset.empty:
#             return "No events registered."
#         return subset[["event_id", "event_type", "risk_score", "source"]].to_string(index=False)

#     def get_entity_graph_metrics(self, entity_id: str) -> dict:
#         if entity_id not in self.graph:
#             return {"error": "Entity not in graph"}
#         neighbors = list(self.graph.neighbors(entity_id))
#         return {
#             "degree": self.graph.degree(entity_id),
#             "connected_events": len(neighbors),
#             "sample_neighbors": neighbors[:5]
#         }

# # Instantiate MCP tool server
# mcp_server = MockMCPToolServer(all_events, G)

# # Load lightweight instruction LLM with explicit torch import
# llm_pipeline = pipeline(
#     "text-generation",
#     model="Qwen/Qwen2.5-1.5B-Instruct",
#     device_map="auto",
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
# )

# def run_mcp_investigation(entity_id: str):
#     # 1. Query read-only MCP tools
#     event_evidence = mcp_server.get_entity_events(entity_id)
#     graph_metrics = mcp_server.get_entity_graph_metrics(entity_id)

#     prompt = f"""<|im_start|>system
# You are a senior SOC and intelligence triage analyst. Analyze evidence retrieved through the MCP tool interface.
# Strict Guidelines:
# 1. Treat model outputs as probabilistic risk signals, not verified facts.
# 2. Group evidence by independent reporting modalities (Network, URL, Financial).
# 3. Specify why multi-modal correlation increases confidence versus isolated alerts.
# 4. Recommend concrete forensic next steps (e.g., PCAP slice, wallet check).<|im_end|>
# <|im_start|>user
# INVESTIGATION TARGET: {entity_id}

# MCP EVIDENCE RETRIEVAL:
# [Observed Events]:
# {event_evidence}

# [Graph Topology Metrics]:
# {json.dumps(graph_metrics, indent=2)}

# Generate a structured intelligence assessment dossier.<|im_end|>
# <|im_start|>assistant
# """
#     output = llm_pipeline(prompt, max_new_tokens=450, do_sample=False)
#     return output[0]["generated_text"].split("<|im_start|>assistant\n")[-1]

# # Run investigation on top correlated entity
# top_target_entity = entity_risk_summary.iloc[0]["entity_id"]
# print(f"Executing MCP Investigation on: {top_target_entity}...\n")
# investigation_report = run_mcp_investigation(top_target_entity)

# print("=" * 80)
# print(f"             FINAL MULTI-MODAL EVIDENCE DOSSIER: {top_target_entity}")
# print("=" * 80)
# print(investigation_report)
# print("=" * 80)

In [ ]:
!pip install -q bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.0 MB/s eta 0:00:00


In [ ]:
# Cell 8 (Fixed): Aggregated MCP Tool Server & Formatted Dossier
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

print("\n--- [Controlled MCP & LLM Investigation Engine (Structured)] ---")

class StructuredMCPToolServer:
    def __init__(self, events_df, graph):
        self.events_df = events_df
        self.graph = graph

    def get_entity_modality_summary(self, entity_id: str) -> dict:
        subset = self.events_df[self.events_df["entity_id"] == entity_id]
        if subset.empty:
            return {"status": "No events registered"}

        summary = {
            "entity_id": entity_id,
            "total_event_count": int(len(subset)),
            "distinct_modalities": subset["event_type"].unique().tolist(),
            "modalities": {}
        }

        for m_type, group in subset.groupby("event_type"):
            top_sample = group.sort_values(by="risk_score", ascending=False).head(3)
            summary["modalities"][m_type] = {
                "event_count": int(len(group)),
                "max_risk_score": float(round(group["risk_score"].max(), 4)),
                "mean_risk_score": float(round(group["risk_score"].mean(), 4)),
                "source_engine": group["source"].iloc[0],
                "top_events": top_sample[["event_id", "risk_score"]].to_dict(orient="records")
            }
        return summary

    def get_entity_graph_topology(self, entity_id: str) -> dict:
        if entity_id not in self.graph:
            return {"error": "Entity not in graph"}
        neighbors = list(self.graph.neighbors(entity_id))
        return {
            "entity_node": entity_id,
            "degree_centrality": self.graph.degree(entity_id),
            "total_connected_events": len(neighbors),
            "sample_connected_events": neighbors[:6]
        }

mcp_server = StructuredMCPToolServer(all_events, G)

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, clean_up_tokenization_spaces=False)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    low_cpu_mem_usage=True
)

llm_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

def run_mcp_investigation(entity_id: str):
    modality_summary = mcp_server.get_entity_modality_summary(entity_id)
    graph_metrics = mcp_server.get_entity_graph_topology(entity_id)

    prompt = f"""<|im_start|>system
You are a senior cyber-intelligence analyst. Synthesize evidence provided via the MCP tools into a structured threat investigation dossier.

Follow this exact structure:
1. THREAT SUMMARY: Overall risk status and why multi-modal correlation matters.
2. MODALITY BREAKDOWN: Analyze Network, Phishing URL, and Financial telemetry signals.
3. GRAPH TOPOLOGY: Interpret connections and structural centrality.
4. FORENSIC ACTIONS: List 3 actionable containment/forensic steps (e.g., PCAP slice, domain sinkhole, transaction freeze).

Do not list raw IDs repeatedly. Provide concise analytical insights.<|im_end|>
<|im_start|>user
TARGET UNDER INVESTIGATION: {entity_id}

[MCP Tool: get_entity_modality_summary]
{json.dumps(modality_summary, indent=2)}

[MCP Tool: get_entity_graph_topology]
{json.dumps(graph_metrics, indent=2)}

Generate the intelligence assessment dossier.<|im_end|>
<|im_start|>assistant
"""
    output = llm_pipeline(
        prompt,
        max_new_tokens=450,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    return output[0]["generated_text"].split("<|im_start|>assistant\n")[-1]

top_target_entity = entity_risk_summary.iloc[0]["entity_id"]
print(f"Executing MCP Investigation on: {top_target_entity}...\n")
investigation_report = run_mcp_investigation(top_target_entity)

print("=" * 80)
print(f"             FINAL MULTI-MODAL EVIDENCE DOSSIER: {top_target_entity}")
print("=" * 80)
print(investigation_report)
print("=" * 80)


--- [Controlled MCP & LLM Investigation Engine (Structured)] ---


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Executing MCP Investigation on: ENTITY_101...



[transformers] Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


             FINAL MULTI-MODAL EVIDENCE DOSSIER: ENTITY_101
### THREAT SUMMARY

**Overall Risk Status:** High due to the high number of financial transactions involving Entity_101, which could indicate fraudulent activities such as money laundering or identity theft. The presence of multiple distinct modalities—financial transactions, network events, and phishing URLs—suggests that there may be a sophisticated attack vector at play.

**Why Multi-Modal Correlation Matters:** By analyzing these different types of data, we can identify patterns and correlations that might help in understanding the nature of the threat. For instance, financial transactions often involve complex financial structures and transactions, making them more susceptible to fraud. Network events, on the other hand, can provide insight into the broader context of an organization's operations. Phishing URLs, while potentially malicious, also offer valuable information about the organization’s security posture and pote

In [ ]:
# Cell 8 (Fixed): Aggregated MCP Tool Server & Formatted Dossier
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

print("\n--- [Controlled MCP & LLM Investigation Engine (Structured)] ---")

class StructuredMCPToolServer:
    def __init__(self, events_df, graph):
        self.events_df = events_df
        self.graph = graph

    def get_entity_modality_summary(self, entity_id: str) -> dict:
        subset = self.events_df[self.events_df["entity_id"] == entity_id]
        if subset.empty:
            return {"status": "No events registered"}

        summary = {
            "entity_id": entity_id,
            "total_event_count": int(len(subset)),
            "distinct_modalities": subset["event_type"].unique().tolist(),
            "modalities": {}
        }

        for m_type, group in subset.groupby("event_type"):
            top_sample = group.sort_values(by="risk_score", ascending=False).head(3)
            summary["modalities"][m_type] = {
                "event_count": int(len(group)),
                "max_risk_score": float(round(group["risk_score"].max(), 4)),
                "mean_risk_score": float(round(group["risk_score"].mean(), 4)),
                "source_engine": group["source"].iloc[0],
                "top_events": top_sample[["event_id", "risk_score"]].to_dict(orient="records")
            }
        return summary

    def get_entity_graph_topology(self, entity_id: str) -> dict:
        if entity_id not in self.graph:
            return {"error": "Entity not in graph"}
        neighbors = list(self.graph.neighbors(entity_id))
        return {
            "entity_node": entity_id,
            "degree_centrality": self.graph.degree(entity_id),
            "total_connected_events": len(neighbors),
            "sample_connected_events": neighbors[:6]
        }

mcp_server = StructuredMCPToolServer(all_events, G)

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, clean_up_tokenization_spaces=False)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    low_cpu_mem_usage=True
)

llm_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

def run_mcp_investigation(entity_id: str):
    modality_summary = mcp_server.get_entity_modality_summary(entity_id)
    graph_metrics = mcp_server.get_entity_graph_topology(entity_id)

    prompt = f"""<|im_start|>system
You are a senior cyber-intelligence analyst. Synthesize evidence provided via the MCP tools into a concise, structured threat investigation dossier.

Follow this exact structure:
1. THREAT SUMMARY: Overall risk status and why multi-modal correlation matters.
2. MODALITY BREAKDOWN: Brief bullets on Network, URL, and Financial signals.
3. GRAPH TOPOLOGY: Interpret degree centrality and connected subgraphs.
4. FORENSIC ACTIONS: List 3 concrete containment steps (e.g., PCAP slice, domain sinkhole, transaction freeze).

Keep each section concise so the full dossier fits within 500 words.<|im_end|>
<|im_start|>user
TARGET UNDER INVESTIGATION: {entity_id}

[MCP Tool: get_entity_modality_summary]
{json.dumps(modality_summary, indent=2)}

[MCP Tool: get_entity_graph_topology]
{json.dumps(graph_metrics, indent=2)}

Generate the intelligence assessment dossier.<|im_end|>
<|im_start|>assistant
"""
    output = llm_pipeline(
        prompt,
        max_new_tokens=700,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    return output[0]["generated_text"].split("<|im_start|>assistant\n")[-1]

top_target_entity = entity_risk_summary.iloc[0]["entity_id"]
print(f"Executing MCP Investigation on: {top_target_entity}...\n")
investigation_report = run_mcp_investigation(top_target_entity)

print("=" * 80)
print(f"             FINAL MULTI-MODAL EVIDENCE DOSSIER: {top_target_entity}")
print("=" * 80)
print(investigation_report)
print("=" * 80)


--- [Controlled MCP & LLM Investigation Engine (Structured)] ---


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Executing MCP Investigation on: ENTITY_101...



[transformers] Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


             FINAL MULTI-MODAL EVIDENCE DOSSIER: ENTITY_101
**THREAT SUMMARY**

The entity **ENTITY_101** has been identified as having multiple types of network, financial, and URL events with varying degrees of risk scores. The high number of distinct modalities indicates that the entity is likely involved in complex cyber-attacks or malicious activities. The presence of financial transactions suggests potential financial fraud or money laundering, while the high risk score for URLs points to possible phishing attempts or other forms of web-based attacks.

**MODALITY BREAKDOWN**

1. **Financial Transaction**: This event type is highly correlated with high-risk scores, indicating a significant amount of activity related to financial crimes. The high frequency of such transactions could suggest ongoing or planned financial activities by the entity.

2. **Network Event**: The network event modality shows a moderate level of risk, but it's not as high as the financial transaction. This c

In [ ]:
##cell 9
import os
import json
import pickle
import warnings
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pyvis.network import Network
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

warnings.filterwarnings("ignore")

# 1. Load Pre-cached Data & Setup Low-RAM LLM Pipeline
with open("/content/fusion_cache.pkl", "rb") as f:
    cache = pickle.load(f)

all_events = cache["all_events"]
entity_risk_summary = cache["entity_risk_summary"]
G = cache["graph"]

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, clean_up_tokenization_spaces=False)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    low_cpu_mem_usage=True
)
llm_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# 2. Structured MCP Tool Abstraction
class StructuredMCPToolServer:
    def __init__(self, events_df, graph):
        self.events_df = events_df
        self.graph = graph

    def get_entity_modality_summary(self, entity_id: str) -> dict:
        subset = self.events_df[self.events_df["entity_id"] == entity_id]
        if subset.empty:
            return {"status": "No events registered"}
        summary = {
            "entity_id": entity_id,
            "total_event_count": int(len(subset)),
            "distinct_modalities": subset["event_type"].unique().tolist(),
            "modalities": {}
        }
        for m_type, group in subset.groupby("event_type"):
            top_sample = group.sort_values(by="risk_score", ascending=False).head(3)
            summary["modalities"][m_type] = {
                "event_count": int(len(group)),
                "max_risk_score": float(round(group["risk_score"].max(), 4)),
                "mean_risk_score": float(round(group["risk_score"].mean(), 4)),
                "source_engine": group["source"].iloc[0],
                "top_events": top_sample[["event_id", "risk_score"]].to_dict(orient="records")
            }
        return summary

    def get_entity_graph_topology(self, entity_id: str) -> dict:
        if entity_id not in self.graph:
            return {"error": "Entity not in graph"}
        neighbors = list(self.graph.neighbors(entity_id))
        return {
            "entity_node": entity_id,
            "degree_centrality": self.graph.degree(entity_id),
            "total_connected_events": len(neighbors),
            "sample_connected_events": neighbors[:6]
        }

mcp_server = StructuredMCPToolServer(all_events, G)

# 3. Interactive Widget Handlers
entity_dropdown = widgets.Dropdown(
    options=entity_risk_summary["entity_id"].tolist(),
    description='Target Entity:',
    style={'description_width': 'initial'}
)

investigate_button = widgets.Button(
    description='Run MCP Investigation',
    button_style='danger',
    tooltip='Query MCP Tools and Generate AI Dossier',
    icon='shield'
)

metrics_output = widgets.Output()
graph_output = widgets.Output()
dossier_output = widgets.Output()

def render_entity_view(change=None):
    selected_entity = entity_dropdown.value

    with metrics_output:
        clear_output()
        meta = entity_risk_summary[entity_risk_summary["entity_id"] == selected_entity].iloc[0]
        html_content = f"""
        <div style="background-color: #1a1a1a; padding: 12px; border-radius: 8px; color: white; margin-bottom: 10px;">
            <b>Entity ID:</b> <span style="color: #ff4b4b;">{selected_entity}</span> |
            <b>Fused Risk Score:</b> <span style="color: #f59e0b;">{meta['fused_risk_score']:.3f}</span> |
            <b>Total Events:</b> {int(meta['total_events'])} |
            <b>Corroborating Sources:</b> {int(meta['source_count'])}
        </div>
        """
        display(HTML(html_content))

    with graph_output:
        clear_output()
        subset_events = all_events[all_events["entity_id"] == selected_entity].head(25)
        net = Network(height="320px", width="100%", bgcolor="#1e1e1e", font_color="white", cdn_resources='remote')
        net.add_node(selected_entity, label=selected_entity, color="#ff4b4b", size=25, title="Target Entity")

        color_map = {
            "network_event": "#3b82f6",
            "url_event": "#eab308",
            "financial_transaction": "#10b981"
        }

        for _, row in subset_events.iterrows():
            c = color_map.get(row["event_type"], "#9ca3af")
            net.add_node(row["event_id"], label=row["event_id"], color=c, size=15, title=f"Risk: {row['risk_score']:.3f}")
            net.add_edge(selected_entity, row["event_id"], title=row["event_type"])

        net.save_graph("/content/temp_subgraph.html")
        with open("/content/temp_subgraph.html", "r", encoding="utf-8") as f:
            display(HTML(f.read()))

def on_investigate_clicked(b):
    selected_entity = entity_dropdown.value
    with dossier_output:
        clear_output()
        print(f"⏳ Querying MCP Server & generating threat dossier for {selected_entity}...")

        modality_summary = mcp_server.get_entity_modality_summary(selected_entity)
        graph_metrics = mcp_server.get_entity_graph_topology(selected_entity)

        prompt = f"""<|im_start|>system
You are a senior cyber-intelligence analyst. Synthesize evidence provided via the MCP tools into a concise, structured threat investigation dossier.

Follow this exact structure:
1. THREAT SUMMARY: Overall risk status and why multi-modal correlation matters.
2. MODALITY BREAKDOWN: Brief bullets on Network, URL, and Financial signals.
3. GRAPH TOPOLOGY: Interpret degree centrality and connected subgraphs.
4. FORENSIC ACTIONS: List 3 concrete containment steps (e.g., PCAP slice, domain sinkhole, transaction freeze).

Keep each section concise.<|im_end|>
<|im_start|>user
TARGET UNDER INVESTIGATION: {selected_entity}

[MCP Tool: get_entity_modality_summary]
{json.dumps(modality_summary, indent=2)}

[MCP Tool: get_entity_graph_topology]
{json.dumps(graph_metrics, indent=2)}

Generate the intelligence assessment dossier.<|im_end|>
<|im_start|>assistant
"""
        output = llm_pipe(prompt, max_new_tokens=600, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        dossier = output[0]["generated_text"].split("<|im_start|>assistant\n")[-1]

        clear_output()
        display(HTML(f"""
        <div style="background-color: #0f172a; border-left: 4px solid #38bdf8; padding: 15px; border-radius: 4px; color: #f8fafc; font-family: sans-serif; white-space: pre-wrap; line-height: 1.5;">
<h3>🛡️ Intelligence Dossier: {selected_entity}</h3>
{dossier}
        </div>
        """))

# Link controls
entity_dropdown.observe(render_entity_view, names='value')
investigate_button.on_click(on_investigate_clicked)

# Display Complete UI in Colab Output Cell
display(widgets.HBox([entity_dropdown, investigate_button]))
display(metrics_output)
display(graph_output)
display(dossier_output)

# Initial render
render_entity_view()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Output()

Output()

Output()

In [ ]:
# Node Color Key:

# 🔴 Red Node: The target entity under investigation (ENTITY_48).

# 🔵 Blue Nodes: Network NetFlow anomaly events (NET_*).

# 🟡 Yellow Nodes: Phishing & malicious URL events (URL_*).

# 🟢 Green Nodes: Financial fraud transaction events (TX_*).

# Hover Details: Hover over any node to view its individual model risk score.

# Generate the LLM Threat Dossier: Click the red "Run MCP Investig..." button. The Qwen-0.5B model will query the read-only MCP tool endpoints and stream the formatted intelligence dossier directly below the graph canvas.